# 01 · nexus inputs

**Purpose.** Check what each nexus sample really contains before it enters IC.

**Inputs.** `$NEXUS_DIR/{Kr,bb}/<sample>_<i>.next.h5` (`/MC/configuration`, `/MC/hits`, `/MC/sns_response`).

**Outputs.** None.

**What is checked.** pressure, generator and region from the stored macro; deposited energy per event (41.5 keV for Kr-83m, 2458 keV for 0νββ); spatial distribution of the deposits; whether nexus already recorded S1 light on the sensors (`/MC/sns_response`).

**Next step.** `02_light_tables`.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

In [ ]:
SAMPLE = "20250624_Next100_1Kev_ELoff_Kr"   # or 20250626_Next100_50Kev_ELoff_Kr, 20250205_Next100_bb_300ev_ELoff
f = paths.nexus_files(SAMPLE)[0]; print(f)
pd.Series(checks.nexus_config(f))

In [ ]:
hits = pd.read_hdf(f, "/MC/hits", start=0, stop=400_000)
E = hits.groupby("event_id").energy.sum() * 1e3   # keV (nexus energy in MeV)
plt.hist(E, 100); plt.xlabel("deposited energy per event (keV)"); plt.ylabel("events")
print(f"{len(E)} events read; median {E.median():.2f} keV")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].hist2d(hits.x, hits.y, 100, cmin=1); ax[0].set_xlabel("x (mm)"); ax[0].set_ylabel("y (mm)"); ax[0].set_aspect("equal")
ax[1].hist(hits.z, 100); ax[1].set_xlabel("z (mm)")
plt.tight_layout()

### Did nexus record S1 light on the sensors?
If `/MC/sns_response` has charge in the S1 window for the fibers, nexus simulated the scintillation photons itself.
That would give a real fiber S1 (unlike the detsim placeholder, see `03_detsim`).

In [ ]:
sns = pd.read_hdf(f, "/MC/sns_positions")
resp = pd.read_hdf(f, "/MC/sns_response")
resp = resp.merge(sns[["sensor_id", "sensor_name"]], on="sensor_id", how="left")
print(sns.sensor_name.value_counts())
per_type = resp.groupby(["sensor_name"]).charge.sum() / resp.event_id.nunique()
print("mean detected charge per event, by sensor type:"); print(per_type)